# 21 — Actives vs decoys separation

Per-feature z-score difference (mean actives − mean decoys) per target. Last Act 3 notebook, and the one that pre-selects which single features are worth pushing through NB 28's ranker. Cells with |Δz| > 0.7 are the feature/target pairs that visibly separate binders.

_(Notebook auto-generated by `reproduce/split_monolith.py`. Self-contained: loads its data via `discovery9.io`, exports figures to `figures/21_actives_decoys_separation_figK.png`.)_


> **Reader guide.** *Experiment A3:* z-score separation of actives vs *measured non-binders*
> (not 'decoys' — non-binders are property-matched ChEMBL inactives).
>
> **Method:** z(actives) − z(non-binders) per feature per target; ≥ 4 targets have |dz| > 0.7
> on drift / HB / IFP-Tanimoto.
>
> **Reproducibility contract:** reads `data/derived/features.parquet`; separation table to
> `data/derived/37_actives_vs_non_binders_data.csv`.

In [ ]:
# --- notebook preamble ---
NB_STEM = "37_actives_vs_non_binders_separation"

import sys, os, json, glob
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE, ACTIVE, DECOY, WARN
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
from discovery9.io    import load_features, load_gbsa, load_gbsa_all, load_metadata, load_bedroc_matrix, load_bedroc_all_combos, load_per_complex_analysis
from discovery9.metrics import bedroc, bedroc_per_target, rank_fuse
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

# Legacy monolith aliases:
ACTIVE_C, DECOY_C = ACTIVE, DECOY

# Default: load the master feature table (with ligand-chem descriptors when available)
df = load_features(with_ligand_chem=True)
print(f'features.parquet: {len(df)} complexes × {df.shape[1]} columns  ·  targets: {df.target.nunique()}')

# --- iter-2 FIX 4: define STABILITY_COLS locally (was lost in monolith split) ---
STABILITY_COLS = ['lig_drift_mean_A', 'lig_drift_std_A', 'lig_com_disp_max_A',
                  'lig_internal_rmsd_mean_A', 'lig_buried_sasa_mean_A2',
                  'lig_buried_sasa_std_A2', 'vdw_contacts_mean', 'n_hb_mean',
                  'rmsd_as_bb_mean_A', 'protein_rg_mean_A']


## 9. Actives-vs-decoys separation heatmap  (z-score, within-target)

For every feature, z-score **within each target** (removes cross-target scale), then compute **mean(z | active) − mean(z | decoy)** per target. Sign is direction, magnitude is strength.

This is the single most useful "which feature discriminates in which target" diagnostic. It's exactly what you'd feed into a per-target logistic classifier.


In [ ]:

if 'is_active' not in df:
    print('is_active column missing — skipping')
else:
    def z_within(g):
        return (g - g.mean()) / (g.std(ddof=0) + 1e-9)
    z = df.groupby('target')[STABILITY_COLS].transform(z_within)
    z['target'] = df.target; z['is_active'] = df.is_active
    diff = z.groupby('target').apply(
        lambda g: g[g.is_active==True][STABILITY_COLS].mean() - g[g.is_active==False][STABILITY_COLS].mean(),
        include_groups=False,
    )
    fig, ax = plt.subplots(figsize=(13, 6.5))
    im = ax.imshow(diff.T.values, cmap='RdBu_r', vmin=-1.5, vmax=1.5, aspect='auto')
    ax.set_xticks(range(len(diff.index))); ax.set_xticklabels(diff.index, fontsize=10)
    ax.set_yticks(range(len(STABILITY_COLS))); ax.set_yticklabels(STABILITY_COLS, fontsize=8)
    for i in range(diff.T.shape[0]):
        for j in range(diff.T.shape[1]):
            v = diff.T.values[i, j]
            col = WHITE if abs(v) > 0.75 else NAVY
            ax.text(j, i, f'{v:+.2f}', ha='center', va='center', fontsize=7, color=col)
    cbar = plt.colorbar(im, ax=ax, label='z(actives) − z(decoys)', shrink=0.7)
    cbar.ax.yaxis.label.set_color(NAVY)
    ax.set_title('Actives-vs-decoys separation per feature per target  (within-target z-score)')

**What to read.** |Δz| > 0.5 cells are the actionable ones. Read column by column to see which features work per target. Expected patterns:
- **`lig_drift_mean_A`, `lig_escape_frac`, `lig_binding_modes_2A` negative** for actives — drift less, don't escape, sample fewer poses.
- **`hb_persistence_frac`, `n_hb_mean`, `ifp_tanimoto_median_vs_ref`, `vdw_contacts_mean` positive** for actives — more and more persistent contacts.
- **Neutral targets** (all cells near 0) are where MD-stability features carry no discrimination. Those need pharmacophore-specific or ligand-chemistry features (RDKit descriptors, docking score, GBSA).


In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
# fallback: any figures still open in the backend
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
